In [1]:
import duckdb

In [2]:
conn = duckdb.connect('../data/warehouse/electronics_sales.duckdb')

In [3]:
conn.execute("SHOW TABLES").fetchall()

[('customer_recurrence',),
 ('customer_recurrency',),
 ('recurring_customers_by_region',),
 ('revenue_seasonality',),
 ('sales',),
 ('sales_per_place',),
 ('sales_per_region',),
 ('top_categories',)]

In [4]:
df = conn.execute("SELECT * FROM sales").fetchdf()

In [5]:
df.columns


Index(['order_id', 'customer_id', 'customer_name', 'customer_segment',
       'customer_type', 'first_purchase_date', 'last_purchase_date',
       'product_id', 'product_name', 'category', 'sub_category', 'brand',
       'order_date', 'quantity', 'unit_price', 'discount_pct', 'sales_channel',
       'payment_method', 'sales_rep', 'region', 'operating_expenses',
       'cash_balance', 'debt_balance', 'monthly_burn', 'churn_flag',
       'gross_revenue', 'net_revenue', 'cost_of_goods_sold', 'gross_profit',
       'operational_profit', 'order_date_month', 'order_date_month_name',
       'order_date_year', 'order_date_quarter', 'first_purchase_date_month',
       'first_purchase_date_month_name', 'first_purchase_date_year',
       'first_purchase_date_quarter', 'last_purchase_date_month',
       'last_purchase_date_month_name', 'last_purchase_date_year',
       'last_purchase_date_quarter'],
      dtype='str')

In [6]:
print(df['net_revenue'].head())

0     417.1260
1     893.2395
2    1094.7660
3     844.5000
4     316.0840
Name: net_revenue, dtype: float64


In [7]:
df['monthly_burn'].head(5)

0    329.57
1     88.62
2    389.89
3    306.67
4    317.36
Name: monthly_burn, dtype: float64

In [8]:
product_operational_profit = df.groupby('product_name')['operational_profit'].sum().reset_index().sort_values(by='operational_profit', ascending=False)
print(product_operational_profit)

          product_name  operational_profit
26      MacBook Pro 14        171099.00950
25      MacBook Air M3        126798.59870
20       Legion Slim 5        100699.75560
11    Galaxy S24 Ultra         96340.07140
37              XPS 13         95165.82550
24      MacBook Air M2         94051.26590
19       Latitude 5440         93415.14290
32        ThinkPad E14         77648.76150
44      iPhone 15 Plus         71147.97200
43           iPhone 15         60873.51600
10          Galaxy S24         54523.36260
16           IdeaPad 5         48711.25860
38           Xiaomi 14         44617.08180
30         Tab Extreme         43376.46425
17         Inspiron 15         39583.58410
42           iPhone 14         38055.54480
13       Galaxy Tab S9         33892.71125
40            iPad Air         31154.13125
41           iPad mini         15916.57300
14    Galaxy Tab S9 FE         14708.88350
39       iPad 10th Gen          4363.03625
31             Tab P12          2107.33675
9          

In [9]:
# Create a view to analyze revenue seasonality

conn.execute("""

CREATE OR REPLACE VIEW revenue_seasonality AS

SELECT
    order_date_year,
    order_date_month,
    order_date_month_name,
    SUM(net_revenue) AS total_revenue
FROM sales
GROUP BY
    order_date_year,
    order_date_month,
    order_date_month_name
ORDER BY
    order_date_year,
    order_date_month

""")

In [10]:
conn.execute("SHOW TABLES").fetchall()

[('customer_recurrence',),
 ('customer_recurrency',),
 ('recurring_customers_by_region',),
 ('revenue_seasonality',),
 ('sales',),
 ('sales_per_place',),
 ('sales_per_region',),
 ('top_categories',)]

In [11]:
conn.execute("""
SELECT *
FROM revenue_seasonality
""").fetchdf()

,order_date_year,order_date_month,order_date_month_name,total_revenue
0,2024,1,Janeiro,224447.1565
1,2024,2,Fevereiro,238581.3435
2,2024,3,Março,193986.0925
3,2024,4,Abril,209394.0595
4,2024,5,Maio,251820.5430
5,2024,6,Junho,221599.5475
6,2024,7,Julho,271436.7590
7,2024,8,Agosto,226709.7030
8,2024,9,Setembro,218799.5340
9,2024,10,Outubro,262265.4845


In [12]:
conn.execute("""
CREATE OR REPLACE VIEW top_categories AS

SELECT
    sub_category,
    ROUND(SUM(net_revenue), 2) AS total_revenue,
    ROUND(SUM(gross_profit), 2) AS total_gross_profit
FROM sales
GROUP BY sub_category

""")

In [13]:
conn.execute("""
SELECT *
FROM top_categories""").fetchdf()

,sub_category,total_revenue,total_gross_profit
0,Peripherals,226264.78,158385.35
1,Tablets,756248.03,378124.01
2,Audio,396887.23,198443.62
3,Laptops,1990441.72,1194265.03
4,Smartphones,2103124.42,841249.77


In [14]:
conn.execute("""
CREATE OR REPLACE VIEW sales_per_region AS
             
SELECT 
region,
COUNT(order_id) AS total_sales,
ROUND(SUM(net_revenue), 2) AS total_revenue,
ROUND(SUM(gross_profit), 2) AS total_gross_profit
FROM sales
GROUP BY region
""")

In [15]:
conn.execute("""
SELECT *
FROM sales_per_region""").fetchdf()

,region,total_sales,total_revenue,total_gross_profit
0,Central,1412,1094129.95,553682.96
1,West,1419,1115827.11,564100.91
2,East,1358,1056781.99,539824.70
3,North,1434,1163648.12,587231.02
4,South,1377,1042579.02,525628.18


In [16]:
conn.execute("""
CREATE OR REPLACE VIEW customer_recurrence AS 
SELECT customer_id, COUNT(order_id) AS total_orders
FROM sales
GROUP BY customer_id
""").fetchdf()

,Count


In [17]:
conn.execute("""
             SELECT *
             FROM customer_recurrence
             """).fetchdf()

,customer_id,total_orders
0,C1002,4
1,C1692,6
2,C1596,6
3,C0009,6
4,C0008,3
...,...,...
1722,C0354,1
1723,C0066,2
1724,C1324,1
1725,C1248,1


In [18]:
conn.execute("""
CREATE OR REPLACE VIEW recurring_customers_by_region AS

WITH customer_region AS (
    SELECT DISTINCT
        customer_id,
        region
    FROM sales
)

SELECT
    r.customer_id,
    r.total_orders,
    cr.region
FROM customer_recurrence r
LEFT JOIN customer_region cr
    ON r.customer_id = cr.customer_id
ORDER BY cr.region ASC;
    """).fetchdf()

,Count


In [19]:
conn.execute("""
             SELECT *
             FROM recurring_customers_by_region
             """).fetchdf()

,customer_id,total_orders,region
0,C1011,1,Central
1,C0480,3,Central
2,C0274,7,Central
3,C0435,4,Central
4,C0399,4,Central
...,...,...,...
4817,C0991,3,West
4818,C0154,2,West
4819,C0687,5,West
4820,C0726,5,West


In [21]:
conn.execute(""" 
             CREATE OR REPLACE VIEW general_kpis AS
             SELECT
             ROUND(SUM(net_revenue), 2) AS total_revenue,
             ROUND(SUM(gross_profit), 2) AS total_gross_profit,
             ROUND(AVG(net_revenue), 2) AS avg_ticket,
             SUM(quantity) AS total_units_sold
             FROM sales
             """).fetchdf()

,Count


In [22]:
conn.close()